In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

class ProbingHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super(ProbingHead, self).__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim,2)
        )

    def forward(self, h):
        return self.mlp(h)

class HallucinationDetector(nn.Module):
    def __init__(self, num_layers, hidden_size, hidden_dim=256):
        super(HallucinationDetector, self).__init__()
        self.num_layers = num_layers
        self.probing_heads = nn.ModuleList([
            ProbingHead(hidden_size,hidden_dim) for _ in range(num_layers)
        ])

    def forward(self, hidden_states):
        logits = []
        for l in range(self.num_layers):
            logits_l = self.probing_heads[l](hidden_states[l])
            logits.append(logits_l)
        logits = torch.stack(logits, dim=0)
        logits = logits.mean(dim=0)
        z_i = F.softmax(logits, dim=-1)
        return z_i

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, probs, targets):
        pt = probs[:, :, 1]
        targets = targets.float()
        loss = - (1 - pt) ** self.gamma * torch.log(pt+1e-8)*targets
        return loss.mean()

if __name__ == "__main__":

    model_name = "Qwen/Qwen3-1.7B"
    model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    prompt = "What is the capital of Mars?"
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.hidden_states

    hidden_states = hidden_states[1:]
    num_layers = len(hidden_states)
    hidden_size = hidden_states[0].shape[-1]

    detector = HallucinationDetector(num_layers=num_layers, hidden_size=hidden_size)
    probs = detector(hidden_states)

    print("hallucination probabilities per token:")
    print(probs[0,:, 1])


In [ ]:

import json
import torch
from torch.utils.data import Dataset
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForCausalLM

class HallucinationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            'input_ids': ex['input_ids'],
            'attention_mask': ex['attention_mask'],
            'labels': ex['labels']
        }

class DatasetBuilder:
    def __init__(self, model_name="Qwen/Qwen2.5-1.5B", max_length=128, device="cuda"):
        self.max_length = max_length
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
        self.model.eval()
        self.scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    def generate_response(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(**inputs, max_length=self.max_length)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def compute_rouge_l_f1(self, prediction, reference):
        scores = self.scorer.score(reference, prediction)
        return scores['rougeL'].fmeasure

    def find_hallucination_point(self, prompt, gold):

        gold_tokens = self.tokenizer(gold, return_tensors="pt", add_special_tokens=False).input_ids[0]
        best_i, best_score = 0, float('-inf')

        prompt_tokens = self.tokenizer(prompt, return_tensors="pt").input_ids[0]
        for i in range(len(gold_tokens) + 1):
            input_ids = torch.cat([prompt_tokens, gold_tokens[:i]])
            inputs = {'input_ids': input_ids.unsqueeze(0), 'attention_mask': torch.ones(1, input_ids.size(0), dtype=torch.long)}
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
            # logits: [1, seq_len, vocab]
            logits = outputs.logits[0]
            score = 0.0

            for j, token_id in enumerate(gold_tokens[i:]):

                pos = input_ids.size(0) - 1
                log_prob = torch.log_softmax(logits[pos], dim=-1)[token_id]
                score += log_prob.item()

            if score > best_score:
                best_score, best_i = score, i
        return best_i

    def build(self, qna_list, f1_correct=0.8, f1_incorrect=0.2):
        examples = []
        for item in qna_list:
            prompt = item.get('question')
            answers = item.get('answers') or item.get('answer')
            if not answers:
                continue
            gold = answers[0] if isinstance(answers, list) else answers

            pred = self.generate_response(prompt)
            f1 = self.compute_rouge_l_f1(pred, gold)

            tokens = self.tokenizer(pred, return_tensors="pt", padding='max_length',
                                     truncation=True, max_length=self.max_length)
            seq_len = tokens.input_ids.size(1)
            labels = torch.zeros(seq_len, dtype=torch.long)

            if f1 >= f1_correct:
                split = 'correct'
            elif f1 <= f1_incorrect:
                split = 'hallucinated'
                idx_h = self.find_hallucination_point(prompt, gold)
                if idx_h < seq_len:
                    labels[idx_h] = 1
            else:
                continue

            examples.append({
                'input_ids': tokens.input_ids.squeeze(0),
                'attention_mask': tokens.attention_mask.squeeze(0),
                'labels': labels,
                'split': split
            })


        correct = [e for e in examples if e['split']=='correct']
        hallu = [e for e in examples if e['split']=='hallucinated']
        N = min(len(correct), len(hallu))
        final = correct[:N] + hallu[:N]
        return HallucinationDataset(final)

if __name__ == "__main__":
    try:
        qna = json.load(open("P495.train.json"))
    except FileNotFoundError:
        raise RuntimeError("entity_questions.json nahi mila. Download kar phle bhosdike")

    builder = DatasetBuilder()
    dataset = builder.build(qna)
    torch.save(dataset, "hallucination_dataset.pt")
